In [1]:
import pandas as pd

In [2]:
#load interactions used in the project, the table with all interactions extracted from databases and the table with all proteins
df_used_interactions=pd.read_csv("../data/HumanOISKS.txt",sep=";", header=None, names=["protein1", "protein2"])
df_all_interactions=pd.read_excel("../data/Interactions_1.xlsx")
df_all_proteins=pd.read_excel("../data/Proteins1.xlsx")

In [3]:
# map the proteins inside interactions
dict_map = df_all_proteins.set_index("HGNC Symbol")["Internal ID"]

df_used_interactions["Source ID"] = df_used_interactions["protein1"].map(dict_map)
df_used_interactions["Target ID"] = df_used_interactions["protein2"].map(dict_map)

In [4]:
#merge the interactions used
rez = df_all_interactions.merge(
    df_used_interactions[["Source ID", "Target ID", "protein1","protein2"]],
    left_on=["Internal ID Source", "Internal ID Target"],
    right_on=["Source ID", "Target ID"],
    how="inner"
)

In [5]:
#exclude one database at a time
rez_OmniPath = rez[rez[["InnateDB.Is In InnateDB", "STRING.Is In STRING", "KEGG.Is In KEGG", "SIGNOR.Is In SIGNOR"]].any(axis=1)]
rez_InnateDB = rez[rez[["OmniPath.Is in OmniPath", "STRING.Is In STRING", "KEGG.Is In KEGG", "SIGNOR.Is In SIGNOR"]].any(axis=1)]
rez_KEGG = rez[rez[["InnateDB.Is In InnateDB", "STRING.Is In STRING", "OmniPath.Is in OmniPath", "SIGNOR.Is In SIGNOR"]].any(axis=1)]
rez_STRING = rez[rez[["InnateDB.Is In InnateDB", "OmniPath.Is in OmniPath", "KEGG.Is In KEGG", "SIGNOR.Is In SIGNOR"]].any(axis=1)]
rez_SIGNOR = rez[rez[["InnateDB.Is In InnateDB", "STRING.Is In STRING", "KEGG.Is In KEGG", "OmniPath.Is in OmniPath"]].any(axis=1)]

print("Interactions in rez_OmniPath", len(rez_OmniPath))
print("Interactions in rez_InnateDB", len(rez_InnateDB))
print("Interactions in rez_KEGG", len(rez_KEGG))
print("Interactions in rez_STRING", len(rez_STRING))
print("Interactions in rez_SIGNOR", len(rez_SIGNOR))



Interactions in rez_OmniPath 45754
Interactions in rez_InnateDB 63591
Interactions in rez_KEGG 63492
Interactions in rez_STRING 38894
Interactions in rez_SIGNOR 63635


In [6]:
#save the results
def save_txt(name_f,table):
    table[["protein1", "protein2"]].to_csv(
        name_f,
        sep=";",
        index=False,
        header=False
    )

save_txt("../data/excluding_database/Human_without_Omnipath",rez_OmniPath)
save_txt("../data/excluding_database/Human_without_InnateDB",rez_InnateDB)
save_txt("../data/excluding_database/Human_without_KEGG",rez_KEGG)
save_txt("../data/excluding_database/Human_without_STRING",rez_STRING)
save_txt("../data/excluding_database/Human_without_SIGNOR",rez_SIGNOR)
